# Colab Fine-Tuning for Sneaker Labels

This notebook uploads a sneaker dataset, optionally creates train/val/test splits, fine-tunes CLIP, and saves the best checkpoint.

In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git ftfy regex tqdm pillow

In [ ]:
import json
import random
import shutil
import zipfile
from pathlib import Path

import clip
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import files
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B/32"
WORKDIR = Path("/content/sneaker_finetune")
RAW_DATASET_ROOT = WORKDIR / "dataset_raw"
SPLIT_ROOT = WORKDIR / "dataset_split"
ARTIFACTS = WORKDIR / "artifacts"
BEST_CHECKPOINT = ARTIFACTS / "clip_sneaker_best.pt"
FINAL_CHECKPOINT = ARTIFACTS / "clip_sneaker_final.pt"
TRAINING_HISTORY = ARTIFACTS / "training_history.json"

for path in [WORKDIR, RAW_DATASET_ROOT, SPLIT_ROOT, ARTIFACTS]:
    path.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Workdir:", WORKDIR)

Upload either:

- a `.zip` with class folders directly inside it, or
- a `.zip` that already contains `train/`, `val/`, and `test/` folders.

In [ ]:
uploaded = files.upload()
archive_name = next(iter(uploaded.keys()))
archive_path = Path(archive_name)
print("Uploaded:", archive_path)

In [ ]:
for path in [RAW_DATASET_ROOT, SPLIT_ROOT, ARTIFACTS]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive_path, "r") as zf:
    zf.extractall(RAW_DATASET_ROOT)

print("Extracted to", RAW_DATASET_ROOT)

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}


def find_dataset_root(root: Path) -> Path:
    if (root / "train").is_dir():
        return root
    children = [p for p in root.iterdir() if p.is_dir()]
    if len(children) == 1 and (children[0] / "train").is_dir():
        return children[0]
    if len(children) == 1:
        nested_children = [p for p in children[0].iterdir() if p.is_dir()]
        if nested_children:
            return children[0]
    return root


def discover_class_dirs(root: Path):
    return sorted({
        p.parent for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    })


def list_images(class_dir: Path):
    return sorted([
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ])


def split_counts(n: int, train_ratio: float, val_ratio: float, test_ratio: float):
    train_n = int(n * train_ratio)
    val_n = int(n * val_ratio)
    test_n = n - train_n - val_n
    if n >= 3:
        if val_n == 0:
            val_n = 1
            train_n -= 1
        if test_n == 0:
            test_n = 1
            train_n -= 1
    elif n == 2 and train_n == 2:
        train_n = 1
        val_n = 1
        test_n = 0
    train_n = max(train_n, 0)
    val_n = max(val_n, 0)
    test_n = max(test_n, 0)
    return train_n, val_n, test_n


def ensure_split_dataset(input_root: Path, output_root: Path, seed: int = 42):
    dataset_root = find_dataset_root(input_root)
    if (dataset_root / "train").is_dir():
        return dataset_root / "train", dataset_root / "val", dataset_root / "test"

    rng = random.Random(seed)
    class_dirs = discover_class_dirs(dataset_root)
    for class_dir in class_dirs:
        images = list_images(class_dir)
        rng.shuffle(images)
        train_n, val_n, test_n = split_counts(len(images), 0.7, 0.15, 0.15)
        splits = {
            "train": images[:train_n],
            "val": images[train_n:train_n + val_n],
            "test": images[train_n + val_n:train_n + val_n + test_n],
        }
        for split_name, split_images in splits.items():
            for src in split_images:
                dst = output_root / split_name / class_dir.name / src.name
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
    return output_root / "train", output_root / "val", output_root / "test"


train_root, val_root, test_root = ensure_split_dataset(RAW_DATASET_ROOT, SPLIT_ROOT)
print("Train:", train_root)
print("Val:", val_root)
print("Test:", test_root)

In [ ]:
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-6
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 0.001
NUM_WORKERS = 2

print({
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_epochs": WARMUP_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "num_workers": NUM_WORKERS,
})

In [ ]:
def format_class_name(class_dir_name: str) -> str:
    return class_dir_name.replace("_", " ").title()


class SneakerDataset(Dataset):
    def __init__(self, dataset_root: Path, preprocess, class_names=None):
        self.dataset_root = Path(dataset_root)
        self.preprocess = preprocess
        self.image_paths = []
        self.labels = []

        class_dirs = discover_class_dirs(self.dataset_root)
        discovered_names = sorted({class_dir.name for class_dir in class_dirs})
        self.class_names = class_names or discovered_names
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}
        self.class_prompts = [
            f"a photo of {format_class_name(class_name)} sneakers"
            for class_name in self.class_names
        ]

        for class_dir in class_dirs:
            label = self.class_to_idx.get(class_dir.name)
            if label is None:
                continue
            image_files = list_images(class_dir)
            self.image_paths.extend(image_files)
            self.labels.extend([label] * len(image_files))

        if not self.image_paths:
            raise RuntimeError(f"No images found under {self.dataset_root}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        return self.preprocess(image), self.labels[idx]


def create_dataloader(dataset, batch_size, shuffle, num_workers):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=(DEVICE == "cuda"),
    )


def convert_models_to_fp32(model):
    for parameter in model.parameters():
        parameter.data = parameter.data.float()
        if parameter.grad is not None:
            parameter.grad.data = parameter.grad.data.float()


@torch.no_grad()
def evaluate(model, dataloader, class_tokens):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    loss_fn = nn.CrossEntropyLoss()

    for images, labels in dataloader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        logits_per_image, _ = model(images, class_tokens)
        loss = loss_fn(logits_per_image, labels)
        predictions = logits_per_image.argmax(dim=1)

        total_loss += loss.item() * len(images)
        total_correct += (predictions == labels).sum().item()
        total_samples += len(images)

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
    }

In [ ]:
model, preprocess = clip.load(MODEL_NAME, device=DEVICE, jit=False)

train_dataset = SneakerDataset(train_root, preprocess)
val_dataset = SneakerDataset(val_root, preprocess, class_names=train_dataset.class_names)
test_dataset = SneakerDataset(test_root, preprocess, class_names=train_dataset.class_names)

train_loader = create_dataloader(train_dataset, BATCH_SIZE, True, NUM_WORKERS)
val_loader = create_dataloader(val_dataset, BATCH_SIZE, False, NUM_WORKERS)
test_loader = create_dataloader(test_dataset, BATCH_SIZE, False, NUM_WORKERS)

class_tokens = clip.tokenize(train_dataset.class_prompts).to(DEVICE)

if DEVICE == "cpu":
    model.float()
else:
    clip.model.convert_weights(model)

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.98),
    eps=1e-6,
    weight_decay=WEIGHT_DECAY,
)
loss_fn = nn.CrossEntropyLoss()


def get_lr(epoch: int):
    if WARMUP_EPOCHS > 0 and epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    return 1.0


scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)
training_history = []
best_metric = float("inf")

print("Classes:", len(train_dataset.class_names))
print("Train images:", len(train_dataset))
print("Val images:", len(val_dataset))
print("Test images:", len(test_dataset))

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    total_samples = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for images, labels in progress_bar:
        optimizer.zero_grad()
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits_per_image, _ = model(images, class_tokens)
        loss = loss_fn(logits_per_image, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        if DEVICE == "cpu":
            optimizer.step()
        else:
            convert_models_to_fp32(model)
            optimizer.step()
            clip.model.convert_weights(model)

        batch_size = len(images)
        epoch_loss += loss.item() * batch_size
        total_samples += batch_size
        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
        })

    scheduler.step()

    train_loss = epoch_loss / total_samples
    val_metrics = evaluate(model, val_loader, class_tokens)
    monitor_loss = val_metrics["loss"]

    history_item = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "lr": optimizer.param_groups[0]["lr"],
    }
    training_history.append(history_item)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} - "
        f"train_loss={train_loss:.4f} - "
        f"val_loss={val_metrics['loss']:.4f} - "
        f"val_acc={val_metrics['accuracy']:.4f}"
    )

    if monitor_loss < best_metric:
        best_metric = monitor_loss
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": monitor_loss,
                "class_names": train_dataset.class_names,
                "train_root": str(train_root),
                "val_root": str(val_root),
                "test_root": str(test_root),
            },
            BEST_CHECKPOINT,
        )
        print("Saved best checkpoint to", BEST_CHECKPOINT)

test_metrics = evaluate(model, test_loader, class_tokens)

torch.save(
    {
        "epoch": EPOCHS,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": best_metric,
        "class_names": train_dataset.class_names,
        "train_root": str(train_root),
        "val_root": str(val_root),
        "test_root": str(test_root),
    },
    FINAL_CHECKPOINT,
)

with open(TRAINING_HISTORY, "w", encoding="utf-8") as file:
    json.dump(training_history, file, indent=2)

print("Training completed")
print("Best checkpoint:", BEST_CHECKPOINT)
print("Final checkpoint:", FINAL_CHECKPOINT)
print("Test metrics:", test_metrics)

In [ ]:
files.download(str(BEST_CHECKPOINT))
files.download(str(FINAL_CHECKPOINT))
files.download(str(TRAINING_HISTORY))